# BeverageDzAI — serveur Kaggle T4 x2

Ce notebook lance **Qwen2.5-7B/14B-Instruct-AWQ** sur le premier GPU et **BAAI/bge-reranker-v2-m3** sur le second. Il expose les deux services par un tunnel Cloudflare authentifié pour l'application BeverageDzAI locale. Le profil 7B est sélectionné par défaut pour réduire la latence ; le 14B peut être choisi dans la cellule 2.

Dans le panneau **Settings** de Kaggle :

1. Choisissez **Accelerator → GPU T4 x2** (pas P100).
2. Activez **Internet**.
3. Exécutez les cellules dans l'ordre.

La première installation redémarre automatiquement le noyau. Après reconnexion, relancez la cellule 1 une fois, puis continuez. L'URL et la clé changent à chaque session.

**Sécurité — à lire avant de partager ce notebook :** la cellule 5 génère un token d'accès et un lien de téléchargement pour `beverage_gpu_connection.env`. Ce token donne un accès complet à ton serveur d'inférence tant que le tunnel est actif. Garde ce notebook **privé** sur Kaggle, ne committe jamais l'output de la cellule 5, et ne partage jamais le fichier `.env` téléchargé.

In [ ]:
# 1 — Installer un environnement vLLM cohérent avec le pilote Kaggle
import importlib.metadata as md
import os, subprocess, sys, time

def version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

ready = (
    version("vllm") == "0.29.0"
    and version("sentence-transformers") is not None
    and version("fastapi") is not None
)
if not ready:
    print("Installation de vLLM et sélection automatique du backend CUDA…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
    subprocess.run(
        ["uv", "pip", "install", "--system", "--torch-backend=auto",
         "vllm==0.29.0", "sentence-transformers", "fastapi",
         "uvicorn", "httpx", "requests"],
        check=True,
    )
    print("Installation terminée. Redémarrage automatique du noyau…")
    time.sleep(3)
    os.kill(os.getpid(), 9)
else:
    print("Dépendances prêtes. vLLM :", version("vllm"), "| torch :", version("torch"))

## 2 — Vérifier les deux GPU

Le notebook s'arrête immédiatement si Kaggle n'a pas fourni deux T4.

In [ ]:
# 2 — Configuration : GPU 0 = Qwen, GPU 1 = reranker
MODEL_SIZE = "7B"  #@param ["7B", "14B"]
import os, re, secrets, subprocess, sys, time, requests, torch
from pathlib import Path
from IPython.display import FileLink, display

if not torch.cuda.is_available():
    raise RuntimeError("GPU absent. Choisissez Accelerator > GPU T4 x2 dans Settings.")
if torch.cuda.device_count() < 2:
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    raise RuntimeError(f"Deux GPU sont requis. GPU détectés : {names}. Sélectionnez GPU T4 x2.")

GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(2)]
if not all("T4" in name.upper() for name in GPU_NAMES):
    raise RuntimeError(f"Ce notebook est validé pour T4 x2, GPU reçus : {GPU_NAMES}")

if MODEL_SIZE not in {"7B", "14B"}:
    raise ValueError("MODEL_SIZE doit être '7B' ou '14B'.")
MODEL_ID = f"Qwen/Qwen2.5-{MODEL_SIZE}-Instruct-AWQ"
SERVED_MODEL = "beverage-qwen"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
MAX_MODEL_LEN = 8192 if MODEL_SIZE == "7B" else 6144
GPU_UTIL = "0.88" if MODEL_SIZE == "7B" else "0.95"
API_TOKEN = secrets.token_urlsafe(32)

# Facultatif : ajoutez HF_TOKEN dans Add-ons > Secrets pour accélérer les téléchargements.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

print("GPU 0 — génération :", GPU_NAMES[0])
print("GPU 1 — reranking  :", GPU_NAMES[1])
print("Modèle :", MODEL_ID)

## 3 — Démarrer Qwen sur le GPU 0

Le premier téléchargement peut prendre plusieurs minutes. Attendez `vLLM prêt`.

In [ ]:
# 3 — Serveur vLLM isolé sur le premier T4
if "vllm_process" in globals() and vllm_process.poll() is None:
    vllm_process.terminate()
    vllm_process.wait(timeout=20)

vllm_env = os.environ.copy()
vllm_env["CUDA_VISIBLE_DEVICES"] = "0"
vllm_env["HF_HOME"] = "/kaggle/working/huggingface-cache"
if HF_TOKEN:
    vllm_env["HF_TOKEN"] = HF_TOKEN

vllm_log = open("/tmp/beverage-vllm.log", "w")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", SERVED_MODEL,
    "--host", "127.0.0.1", "--port", "8001",
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", GPU_UTIL,
    "--max-num-seqs", "1",
    "--enforce-eager",
]
vllm_process = subprocess.Popen(vllm_cmd, env=vllm_env, stdout=vllm_log, stderr=subprocess.STDOUT)
deadline = time.time() + 1800
while time.time() < deadline:
    if vllm_process.poll() is not None:
        print(Path("/tmp/beverage-vllm.log").read_text(errors="replace")[-10000:])
        raise RuntimeError("vLLM s'est arrêté. Le diagnostic est affiché ci-dessus.")
    try:
        if requests.get("http://127.0.0.1:8001/v1/models", timeout=5).ok:
            print("vLLM prêt sur le GPU 0.")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM n'a pas démarré dans les 30 minutes.")

## 4 — Démarrer le reranker sur le GPU 1

La passerelle authentifiée expose `/rerank`, `/health` et les routes OpenAI `/v1/...`.

**Correction apportée ici :** `/health` ne renvoie plus une chaîne codée en dur (`"cuda:0 (GPU physique 1)"`). Il interroge réellement `torch.cuda` au moment de l'appel et renvoie aussi la valeur effective de `CUDA_VISIBLE_DEVICES` vue par le sous-processus, pour qu'un succès affiché corresponde toujours à un succès vérifié — et pas à une supposition sur la façon dont l'isolation *devrait* se comporter.

In [ ]:
# 4 — Passerelle + BGE isolés sur le second T4
gateway_code = r'''
import os
import httpx
import torch
from fastapi import FastAPI, Request, Response
from pydantic import BaseModel
from sentence_transformers import CrossEncoder

TOKEN = os.environ["BEVERAGE_GPU_TOKEN"]
RERANKER_MODEL = os.environ.get("BEVERAGE_RERANKER_MODEL", "BAAI/bge-reranker-v2-m3")
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
reranker = CrossEncoder(
    RERANKER_MODEL, device="cuda", max_length=512,
    model_kwargs={"torch_dtype": torch.float16},
)

@app.middleware("http")
async def authenticate(request: Request, call_next):
    if request.headers.get("authorization") != f"Bearer {TOKEN}":
        return Response("Unauthorized", status_code=401)
    return await call_next(request)

class RerankRequest(BaseModel):
    model: str
    query: str
    documents: list[str]
    top_n: int = 10

@app.get("/health")
def health():
    # Vérification réelle plutôt qu'une chaîne fixe : si l'isolation CUDA_VISIBLE_DEVICES
    # a échoué silencieusement, ce endpoint doit le révéler au lieu de le masquer.
    device_index = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(device_index)
    visible = os.environ.get("CUDA_VISIBLE_DEVICES", "(non défini)")
    device_count_visible = torch.cuda.device_count()
    mem_allocated_gb = round(torch.cuda.memory_allocated(device_index) / 1e9, 2)
    return {
        "status": "ok",
        "reranker": RERANKER_MODEL,
        "cuda_visible_devices_env": visible,
        "torch_device_count_visible": device_count_visible,
        "torch_current_device_index": device_index,
        "torch_current_device_name": device_name,
        "reranker_gpu_memory_allocated_gb": mem_allocated_gb,
    }

@app.post("/rerank")
def rerank(payload: RerankRequest):
    if not payload.documents:
        return {"results": []}
    pairs = [(payload.query, doc) for doc in payload.documents]
    try:
        scores = reranker.predict(
            pairs, batch_size=16, show_progress_bar=False,
            activation_fn=torch.nn.Sigmoid(),
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        scores = reranker.predict(
            pairs, batch_size=4, show_progress_bar=False,
            activation_fn=torch.nn.Sigmoid(),
        )
    order = sorted(range(len(scores)), key=lambda i: float(scores[i]), reverse=True)[:payload.top_n]
    return {"results": [{"index": i, "relevance_score": float(scores[i])} for i in order]}

@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy_vllm(path: str, request: Request):
    body = await request.body()
    async with httpx.AsyncClient(timeout=900) as client:
        upstream = await client.request(
            request.method, f"http://127.0.0.1:8001/v1/{path}",
            content=body,
            headers={"content-type": request.headers.get("content-type", "application/json")},
        )
    return Response(
        content=upstream.content, status_code=upstream.status_code,
        media_type=upstream.headers.get("content-type"),
    )
'''
Path("/tmp/beverage_gateway.py").write_text(gateway_code)

if "gateway_process" in globals() and gateway_process.poll() is None:
    gateway_process.terminate()
    gateway_process.wait(timeout=20)

gateway_env = os.environ.copy()
gateway_env["CUDA_VISIBLE_DEVICES"] = "1"
gateway_env["BEVERAGE_GPU_TOKEN"] = API_TOKEN
gateway_env["BEVERAGE_RERANKER_MODEL"] = RERANKER_MODEL
gateway_env["HF_HOME"] = "/kaggle/working/huggingface-cache"
if HF_TOKEN:
    gateway_env["HF_TOKEN"] = HF_TOKEN
gateway_log = open("/tmp/beverage-gateway.log", "w")
gateway_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "beverage_gateway:app",
     "--app-dir", "/tmp", "--host", "127.0.0.1", "--port", "8000"],
    env=gateway_env, stdout=gateway_log, stderr=subprocess.STDOUT,
)
headers = {"Authorization": f"Bearer {API_TOKEN}"}
deadline = time.time() + 900
while time.time() < deadline:
    if gateway_process.poll() is not None:
        print(Path("/tmp/beverage-gateway.log").read_text(errors="replace")[-10000:])
        raise RuntimeError("La passerelle s'est arrêtée. Le diagnostic est affiché ci-dessus.")
    try:
        response = requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=5)
        if response.ok:
            info = response.json()
            print("Passerelle et reranker prêts sur le GPU 1 :", info)
            # Garde-fou : si le sous-processus voit malgré tout plus d'un GPU, ou si
            # CUDA_VISIBLE_DEVICES n'a pas la valeur attendue, on arrête tout de suite
            # plutôt que de continuer sur une isolation GPU non garantie.
            if info["cuda_visible_devices_env"] != "1":
                raise RuntimeError(
                    "Isolation GPU suspecte : CUDA_VISIBLE_DEVICES="
                    + repr(info["cuda_visible_devices_env"])
                    + " au lieu de '1'. Le reranker pourrait tourner sur le même GPU que vLLM."
                )
            if info["torch_device_count_visible"] != 1:
                raise RuntimeError(
                    "Le sous-processus voit "
                    + str(info["torch_device_count_visible"])
                    + " GPU au lieu de 1 — l'isolation par CUDA_VISIBLE_DEVICES n'est pas effective."
                )
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("La passerelle n'a pas démarré dans les 15 minutes.")

## 5 — Créer le tunnel et télécharger la connexion

Le lien généré permet de télécharger `beverage_gpu_connection.env` depuis l'onglet Kaggle.

**Rappel sécurité :** ce fichier contient un token qui donne un accès complet au serveur tant que le tunnel vit. Ne committe pas l'output de cette cellule, ne partage pas le fichier téléchargé.

In [ ]:
# 5 — Tunnel Cloudflare authentifié
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    tunnel_process.wait(timeout=20)

cloudflared = Path("/tmp/cloudflared")
if not cloudflared.exists():
    download = requests.get(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        timeout=180,
    )
    download.raise_for_status()
    cloudflared.write_bytes(download.content)
    cloudflared.chmod(0o755)

tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError("URL Cloudflare introuvable. Vérifiez que Internet est activé puis relancez cette cellule.")

os.chdir("/kaggle/working")
connection_file = Path("beverage_gpu_connection.env")
connection_file.write_text(
    f"VLLM_BASE_URL={public_url}/v1\n"
    f"RERANKER_BASE_URL={public_url}\n"
    f"VLLM_API_KEY={API_TOKEN}\n"
    f"RERANKER_API_KEY={API_TOKEN}\n"
)
print("URL temporaire créée :", public_url)
print("Cliquez sur le lien suivant pour télécharger le fichier privé :")
display(FileLink(str(connection_file), result_html_prefix=""))

## 6 — Valider réellement les deux services

Ce test confirme maintenant explicitement, via `/health`, que le reranker tourne bien sur un GPU isolé du GPU de génération — au lieu de se fier à une chaîne qui l'affirmait sans le vérifier.

In [ ]:
# 6 — Test de bout en bout
health = requests.get(public_url + "/health", headers=headers, timeout=60)
health.raise_for_status()
health_info = health.json()

# Vérification explicite de l'isolation GPU avant de continuer.
assert health_info["cuda_visible_devices_env"] == "1", (
    f"Isolation GPU non confirmée : {health_info}"
)
assert health_info["torch_device_count_visible"] == 1, (
    f"Le reranker voit plus d'un GPU : {health_info}"
)

rerank_test = requests.post(
    public_url + "/rerank", headers=headers, timeout=180,
    json={
        "model": RERANKER_MODEL,
        "query": "citrus oxidation",
        "documents": ["citral degrades by oxidation", "sugar sweetness"],
        "top_n": 2,
    },
)
rerank_test.raise_for_status()
chat_test = requests.post(
    public_url + "/v1/chat/completions", headers=headers, timeout=300,
    json={
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Réponds uniquement: OK"}],
        "temperature": 0.0,
        "max_tokens": 8,
    },
)
if not chat_test.ok:
    print(chat_test.text[:4000])
chat_test.raise_for_status()
print("SERVEUR KAGGLE VALIDÉ — isolation GPU confirmée par /health")
print("État :", health_info)
print("Reranker :", rerank_test.json())
print("Qwen :", chat_test.json()["choices"][0]["message"]["content"])

## 7 — Garder la session active

Laissez cette cellule tourner pendant l'utilisation de BeverageDzAI. Une session Kaggle GPU reste temporaire et s'arrêtera au plus tard à la limite imposée par Kaggle.

**Correction apportée ici :** la surveillance précédente ne détectait qu'un processus totalement arrêté (`poll() is not None`). Un tunnel ou une passerelle qui restent en vie mais ne répondent plus correctement passaient inaperçus. La boucle interroge maintenant `/health` à intervalle régulier et s'arrête aussi en cas de réponse dégradée ou d'échec de connexion répété — pas seulement en cas de mort du processus.

In [ ]:
# 7 — Maintien et surveillance active des trois processus
print("Serveur actif. Laissez cette cellule tourner pendant la démo.")
CHECK_EVERY_S = 30
MAX_CONSECUTIVE_HEALTH_FAILURES = 3
consecutive_failures = 0

try:
    while True:
        stopped = {
            "vLLM": vllm_process.poll(),
            "passerelle": gateway_process.poll(),
            "tunnel": tunnel_process.poll(),
        }
        if any(code is not None for code in stopped.values()):
            raise RuntimeError(f"Un service s'est arrêté : {stopped}. Consultez /tmp/beverage-*.log")

        try:
            response = requests.get(public_url + "/health", headers=headers, timeout=10)
            response.raise_for_status()
            info = response.json()
            if info.get("cuda_visible_devices_env") != "1" or info.get("torch_device_count_visible") != 1:
                raise RuntimeError(f"Isolation GPU dégradée en cours de session : {info}")
            consecutive_failures = 0
        except requests.RequestException as exc:
            consecutive_failures += 1
            print(f"Avertissement : /health injoignable ({consecutive_failures}/"
                  f"{MAX_CONSECUTIVE_HEALTH_FAILURES}) — {exc}")
            if consecutive_failures >= MAX_CONSECUTIVE_HEALTH_FAILURES:
                raise RuntimeError(
                    "Le tunnel ou la passerelle ne répond plus après plusieurs tentatives, "
                    "alors que les processus sont toujours en vie. Vérifiez la connexion "
                    "réseau ou redémarrez la cellule 5."
                )

        time.sleep(CHECK_EVERY_S)
except KeyboardInterrupt:
    print("Arrêt manuel demandé.")